# One-Dimensional Stromatolite Growth Model

This notebook implements a one-dimensional vertical column model demonstrating microbial growth, EPS production, sediment trapping, burial, and lamination.

The model is deliberately interpretable rather than exhaustive. Each timestep updates one exposed microbial mat. When the mat thickens beyond a lamina threshold, it is buried and a new microbial mat forms above it. Repetition produces a vertical stack of alternating biomass-rich and sediment-rich laminae.

## Model Rules

- Incident light varies annually using a sinusoidal seasonal forcing function.
- Light reaching the active mat is attenuated through the overlying water column, not through the already-built stromatolite column.
- As the stromatolite grows upward, water depth above the mat decreases and light at the active surface can increase.
- Temperature varies annually and modifies biological rates through a bell-shaped suitability response.
- Sediment supply combines baseline supply, slow seasonal variation, daily weather noise, and rare storm pulses.
- Photosynthetic biomass growth follows a Monod-style light response multiplied by temperature suitability.
- Biomass produces EPS, and EPS production is also temperature-limited.
- EPS increases sediment trapping.
- Trapped sediment, new biomass, and EPS increase active mat thickness.
- Burial is triggered by trapped sediment cover or a rapid sediment pulse, after which a new mat is seeded at the surface.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMAGE_DIR = PROJECT_ROOT / "diagrams"
PARAMETER_FILE = DATA_DIR / "1d-parameters.json"

In [ ]:
@dataclass
class ModelParameters1D:
    """Container for the 1-D vertical-column model parameters.

    :param time_steps: Number of explicit timesteps to simulate.
    :param dt_days: Duration of each timestep in days.
    :param initial_mat_thickness_mm: Starting thickness for each newly exposed mat.
    :param lamina_completion_thickness_mm: Legacy thickness threshold retained for comparison.
    :param surface_light: Mean relative incident light above the water column.
    :param light_amplitude: Annual sinusoidal amplitude applied to mean incident light.
    :param light_peak_day: Day of year when incident light reaches its maximum.
    :param min_surface_light: Minimum allowed incident light after seasonal forcing.
    :param light_attenuation_per_mm: Legacy column attenuation coefficient retained for comparison.
    :param water_depth_mm: Water level above the initial stromatolite base.
    :param minimum_water_cover_mm: Minimum water cover retained above the active mat surface.
    :param water_light_attenuation_per_mm: Beer-Lambert attenuation coefficient through water.
    :param mean_temperature_c: Mean annual temperature in degrees Celsius.
    :param temperature_amplitude_c: Annual sinusoidal temperature amplitude in degrees Celsius.
    :param temperature_peak_day: Day of year when temperature reaches its maximum.
    :param optimal_temperature_c: Temperature where biological rates are maximal.
    :param temperature_width_c: Width of the bell-shaped biological temperature response.
    :param days_per_year: Annual period used by seasonal forcing functions.
    :param max_growth_rate_per_day: Maximum photosynthetic growth rate.
    :param half_saturation_light: Light level giving half-maximal growth.
    :param mortality_rate_per_day: Biomass mortality rate in the active surface mat.
    :param buried_mortality_rate_per_day: Additional biomass mortality in buried layers.
    :param eps_production_rate_per_day: EPS production rate per unit biomass.
    :param eps_decay_rate_per_day: EPS decay or compaction rate.
    :param sediment_input_mm_per_day: Baseline potential sediment supplied from the water column.
    :param sediment_seasonal_amplitude: Annual sinusoidal amplitude applied to baseline sediment supply.
    :param sediment_peak_day: Day of year when seasonal sediment supply reaches its maximum.
    :param min_sediment_multiplier: Minimum allowed seasonal sediment multiplier.
    :param storm_probability_per_day: Probability of a rare storm sediment pulse on any simulated day.
    :param storm_sediment_mm_per_day: Mean additional sediment supply during a storm pulse.
    :param storm_noise_fraction: Relative random variation in storm pulse size.
    :param trapping_efficiency: Maximum fraction of supplied sediment retained by EPS.
    :param eps_half_saturation: EPS level giving half-maximal sediment trapping.
    :param biomass_to_thickness: Conversion from biomass growth to thickness.
    :param eps_to_thickness: Conversion from EPS gain to thickness.
    :param sediment_to_thickness: Conversion from trapped sediment to thickness.
    :param burial_sediment_threshold_mm: Cumulative trapped sediment cover needed to bury a mat.
    :param rapid_burial_sediment_threshold_mm: Single-timestep trapped sediment pulse that triggers burial.
    :param new_layer_biomass_seed: Biomass used to seed a new surface mat.
    :param new_layer_eps_seed: EPS used to seed a new surface mat.
    :param random_seed: Seed for reproducible sediment-supply variation.
    :param sediment_noise_fraction: Fractional random variation in sediment supply.
    """

    time_steps: int
    dt_days: float
    initial_mat_thickness_mm: float
    lamina_completion_thickness_mm: float
    surface_light: float
    light_amplitude: float
    light_peak_day: float
    min_surface_light: float
    light_attenuation_per_mm: float
    water_depth_mm: float
    minimum_water_cover_mm: float
    water_light_attenuation_per_mm: float
    mean_temperature_c: float
    temperature_amplitude_c: float
    temperature_peak_day: float
    optimal_temperature_c: float
    temperature_width_c: float
    days_per_year: float
    max_growth_rate_per_day: float
    half_saturation_light: float
    mortality_rate_per_day: float
    buried_mortality_rate_per_day: float
    eps_production_rate_per_day: float
    eps_decay_rate_per_day: float
    sediment_input_mm_per_day: float
    sediment_seasonal_amplitude: float
    sediment_peak_day: float
    min_sediment_multiplier: float
    storm_probability_per_day: float
    storm_sediment_mm_per_day: float
    storm_noise_fraction: float
    trapping_efficiency: float
    eps_half_saturation: float
    biomass_to_thickness: float
    eps_to_thickness: float
    sediment_to_thickness: float
    burial_sediment_threshold_mm: float
    rapid_burial_sediment_threshold_mm: float
    new_layer_biomass_seed: float
    new_layer_eps_seed: float
    random_seed: int
    sediment_noise_fraction: float

In [ ]:
def load_1d_parameters(parameter_path: Path) -> ModelParameters1D:
    """Load model parameters from the baseline JSON file.

    :param parameter_path: Path to a JSON file with a parameters object.
    :return: A populated parameters instance.
    """
    with parameter_path.open() as handle:
        parameter_document = json.load(handle)

    raw_values = {
        name: parameter["value"]
        for name, parameter in parameter_document["parameters"].items()
    }
    raw_values["time_steps"] = int(raw_values["time_steps"])
    raw_values["random_seed"] = int(raw_values["random_seed"])
    return ModelParameters1D(**raw_values)

In [ ]:
def seasonal_light(day: float, params: ModelParameters1D) -> float:
    """Compute seasonally forced surface light for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean light, amplitude, peak day, and minimum light.
    :return: Relative surface light after seasonal forcing and lower clamping.
    """
    phase = 2.0 * math.pi * (day - params.light_peak_day) / params.days_per_year
    forced_light = params.surface_light * (1.0 + params.light_amplitude * math.cos(phase))
    return max(params.min_surface_light, forced_light)

In [ ]:
def seasonal_temperature(day: float, params: ModelParameters1D) -> float:
    """Compute seasonally forced temperature for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean temperature, amplitude, and peak day.
    :return: Temperature in degrees Celsius.
    """
    phase = 2.0 * math.pi * (day - params.temperature_peak_day) / params.days_per_year
    return params.mean_temperature_c + params.temperature_amplitude_c * math.cos(phase)

In [ ]:
def create_layer(thickness_mm: float, biomass: float, eps: float, sediment_mm: float, birth_step: int) -> dict:
    """Create one layer in the vertical stromatolite stack.

    :param thickness_mm: Initial physical layer thickness in millimetres.
    :param biomass: Relative microbial biomass in the layer.
    :param eps: Relative EPS concentration in the layer.
    :param sediment_mm: Sediment thickness contribution already trapped in the layer.
    :param birth_step: Simulation step when the layer first became exposed.
    :return: Mutable layer record used by the timestep update functions.
    """
    return {
        "thickness_mm": thickness_mm,
        "biomass": biomass,
        "eps": eps,
        "sediment_mm": sediment_mm,
        "birth_step": birth_step,
        "burial_step": None,
    }

In [ ]:
def light_at_depth(surface_light: float, attenuation_per_mm: float, depth_mm: float) -> float:
    """Compute light intensity at depth using Beer-Lambert attenuation.

    :param surface_light: Relative light intensity before passing through the attenuating medium.
    :param attenuation_per_mm: Light attenuation coefficient per millimetre.
    :param depth_mm: Path length through the attenuating medium in millimetres.
    :return: Relative light intensity at the requested depth.
    """
    return surface_light * math.exp(-attenuation_per_mm * depth_mm)

In [ ]:
def water_depth_above_mat(column_height_mm: float, params: ModelParameters1D) -> float:
    """Compute water depth above the active mat surface.

    :param column_height_mm: Current stromatolite column height above the initial base.
    :param params: Model parameters controlling water level and minimum water cover.
    :return: Water depth above the active mat in millimetres.
    """
    return max(params.minimum_water_cover_mm, params.water_depth_mm - column_height_mm)

In [ ]:
def light_at_mat(day: float, column_height_mm: float, params: ModelParameters1D) -> dict:
    """Compute light reaching the active mat through overlying water.

    :param day: Day within the seasonal cycle.
    :param column_height_mm: Current stromatolite column height above the initial base.
    :param params: Model parameters controlling incident light and water attenuation.
    :return: Dictionary containing incident light, water depth, and light at the active mat.
    """
    incident_light = seasonal_light(day, params)
    water_depth_mm = water_depth_above_mat(column_height_mm, params)
    mat_light = light_at_depth(
        incident_light,
        params.water_light_attenuation_per_mm,
        water_depth_mm,
    )
    return {
        "incident_light": incident_light,
        "water_depth_above_mat_mm": water_depth_mm,
        "light_at_mat": mat_light,
    }

In [ ]:
def day_of_year(step: int, dt_days: float, days_per_year: float) -> float:
    """Convert a timestep into a repeating day-of-year value.

    :param step: Current simulation step.
    :param dt_days: Duration of each timestep in days.
    :param days_per_year: Number of days in one seasonal cycle.
    :return: Day within the seasonal cycle.
    """
    return (step * dt_days) % days_per_year

In [ ]:
def temperature_response(temperature_c: float, optimal_temperature_c: float, temperature_width_c: float) -> float:
    """Compute biological suitability from temperature.

    :param temperature_c: Current temperature in degrees Celsius.
    :param optimal_temperature_c: Temperature where the multiplier reaches one.
    :param temperature_width_c: Width of the bell-shaped response curve.
    :return: Biological rate multiplier between zero and one.
    """
    scaled_difference = (temperature_c - optimal_temperature_c) / temperature_width_c
    return math.exp(-(scaled_difference ** 2))

In [ ]:
def light_limitation(light: float, half_saturation_light: float) -> float:
    """Compute the Monod-style light limitation factor.

    :param light: Local relative light intensity.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Photosynthetic light limitation factor between zero and one.
    """
    return light / (half_saturation_light + light)

In [ ]:
def monod_growth_rate(light: float, max_growth_rate: float, half_saturation_light: float) -> float:
    """Compute a light-limited photosynthetic growth rate.

    :param light: Local relative light intensity.
    :param max_growth_rate: Maximum possible growth rate per day.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Growth rate per day.
    """
    return max_growth_rate * light_limitation(light, half_saturation_light)

In [ ]:
def trapping_fraction(eps: float, max_efficiency: float, eps_half_saturation: float) -> float:
    """Compute the fraction of supplied sediment trapped by EPS.

    :param eps: Relative EPS concentration in the active mat.
    :param max_efficiency: Maximum possible trapping fraction.
    :param eps_half_saturation: EPS level where trapping reaches half its maximum.
    :return: Fraction of incoming sediment retained by the mat.
    """
    return max_efficiency * eps / (eps_half_saturation + eps)

In [ ]:
def seasonal_sediment_multiplier(day: float, params: ModelParameters1D) -> float:
    """Compute the slow seasonal multiplier for sediment supply.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling sediment seasonality and lower clamping.
    :return: Multiplicative sediment-supply factor.
    """
    phase = 2.0 * math.pi * (day - params.sediment_peak_day) / params.days_per_year
    multiplier = 1.0 + params.sediment_seasonal_amplitude * math.cos(phase)
    return max(params.min_sediment_multiplier, multiplier)

In [ ]:
def daily_sediment_weather_multiplier(noise_fraction: float, rng: np.random.Generator) -> float:
    """Draw a non-negative daily weather multiplier for sediment supply.

    :param noise_fraction: Standard deviation of daily weather variation as a fraction of the mean.
    :param rng: NumPy random generator used for reproducible draws.
    :return: Daily multiplicative sediment-supply factor.
    """
    return max(0.0, rng.normal(loc=1.0, scale=noise_fraction))

In [ ]:
def storm_sediment_pulse(params: ModelParameters1D, rng: np.random.Generator) -> tuple[float, bool]:
    """Draw a rare storm sediment pulse for one timestep.

    :param params: Model parameters controlling storm probability and pulse size.
    :param rng: NumPy random generator used for reproducible storm timing and size.
    :return: Additional sediment supply in millimetres per day and whether a storm occurred.
    """
    storm_probability = min(1.0, params.storm_probability_per_day * params.dt_days)
    storm_today = rng.random() < storm_probability
    if not storm_today:
        return 0.0, False

    pulse = rng.normal(
        loc=params.storm_sediment_mm_per_day,
        scale=params.storm_sediment_mm_per_day * params.storm_noise_fraction,
    )
    return max(0.0, pulse), True

In [ ]:
def variable_sediment_supply(day: float, params: ModelParameters1D, rng: np.random.Generator) -> dict:
    """Combine baseline, seasonal, weather, and storm sediment supply terms.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling all sediment-supply forcing terms.
    :param rng: NumPy random generator used for reproducible stochastic forcing.
    :return: Dictionary containing sediment supply components in millimetres per day.
    """
    seasonal_multiplier = seasonal_sediment_multiplier(day, params)
    weather_multiplier = daily_sediment_weather_multiplier(params.sediment_noise_fraction, rng)
    storm_pulse, storm_today = storm_sediment_pulse(params, rng)
    background_supply = params.sediment_input_mm_per_day * seasonal_multiplier * weather_multiplier
    total_supply = background_supply + storm_pulse

    return {
        "sediment_baseline_mm_per_day": params.sediment_input_mm_per_day,
        "sediment_seasonal_multiplier": seasonal_multiplier,
        "sediment_weather_multiplier": weather_multiplier,
        "storm_sediment_mm_per_day": storm_pulse,
        "storm_today": storm_today,
        "sediment_supply_mm_per_day": total_supply,
    }

In [ ]:
def age_buried_layers(layers: list[dict], params: ModelParameters1D) -> None:
    """Apply biomass decay to all buried layers.

    :param layers: Current vertical stack of layer records.
    :param params: Model parameters controlling timestep length and mortality.
    :return: None. The layer records are updated in place.
    """
    decay = math.exp(-params.buried_mortality_rate_per_day * params.dt_days)
    for layer in layers[:-1]:
        layer["biomass"] *= decay

In [ ]:
def update_active_layer(active_layer: dict, params: ModelParameters1D, step: int, column_height_mm: float, rng: np.random.Generator) -> dict:
    """Advance the exposed microbial mat by one timestep.

    :param active_layer: Mutable record for the current surface mat.
    :param params: Model parameters controlling growth, EPS, sediment trapping, and environmental forcing.
    :param step: Current simulation step.
    :param column_height_mm: Current stromatolite column height above the initial base.
    :param rng: NumPy random generator used for sediment-supply variation.
    :return: Dictionary of timestep diagnostics.
    """
    current_day = day_of_year(step, params.dt_days, params.days_per_year)
    light_forcing = light_at_mat(current_day, column_height_mm, params)
    temperature_today = seasonal_temperature(current_day, params)
    temp_factor = temperature_response(
        temperature_today,
        params.optimal_temperature_c,
        params.temperature_width_c,
    )
    light = light_forcing["light_at_mat"]
    growth_rate = monod_growth_rate(light, params.max_growth_rate_per_day, params.half_saturation_light) * temp_factor
    eps_production_rate = params.eps_production_rate_per_day * temp_factor
    biomass_change = (growth_rate - params.mortality_rate_per_day) * active_layer["biomass"] * params.dt_days
    eps_change = (eps_production_rate * active_layer["biomass"] - params.eps_decay_rate_per_day * active_layer["eps"]) * params.dt_days
    sediment_forcing = variable_sediment_supply(current_day, params, rng)
    supplied_sediment = sediment_forcing["sediment_supply_mm_per_day"] * params.dt_days
    retained_fraction = trapping_fraction(active_layer["eps"], params.trapping_efficiency, params.eps_half_saturation)
    trapped_sediment = supplied_sediment * retained_fraction

    active_layer["biomass"] = max(0.0, active_layer["biomass"] + biomass_change)
    active_layer["eps"] = max(0.0, active_layer["eps"] + eps_change)
    active_layer["sediment_mm"] += trapped_sediment
    active_layer["thickness_mm"] += max(0.0, biomass_change) * params.biomass_to_thickness
    active_layer["thickness_mm"] += max(0.0, eps_change) * params.eps_to_thickness
    active_layer["thickness_mm"] += trapped_sediment * params.sediment_to_thickness

    return {
        "day_of_year": current_day,
        **light_forcing,
        "surface_light": light_forcing["incident_light"],
        "temperature_c": temperature_today,
        "temperature_factor": temp_factor,
        "light": light,
        "light_factor": light_limitation(light, params.half_saturation_light),
        "growth_rate": growth_rate,
        "eps_production_rate": eps_production_rate,
        "biomass_change": biomass_change,
        "eps_change": eps_change,
        **sediment_forcing,
        "supplied_sediment_mm": supplied_sediment,
        "trapped_sediment_mm": trapped_sediment,
        "trapping_fraction": retained_fraction,
    }

In [ ]:
def burial_reason(active_layer: dict, params: ModelParameters1D, diagnostics: dict) -> str | None:
    """Determine whether sediment deposition has buried the active mat.

    :param active_layer: Mutable record for the current surface mat.
    :param params: Model parameters controlling sediment-driven burial.
    :param diagnostics: Timestep diagnostics containing trapped sediment and storm information.
    :return: Burial reason string, or None if the mat remains active.
    """
    if diagnostics["trapped_sediment_mm"] >= params.rapid_burial_sediment_threshold_mm:
        return "rapid_sediment_pulse"
    if active_layer["sediment_mm"] >= params.burial_sediment_threshold_mm:
        return "sediment_cover"
    return None

In [ ]:
def maybe_bury_active_layer(layers: list[dict], params: ModelParameters1D, step: int, diagnostics: dict) -> tuple[bool, str | None]:
    """Bury the active layer if sediment deposition has covered it.

    :param layers: Current vertical stack of layer records.
    :param params: Model parameters controlling sediment-driven burial and new-layer seeding.
    :param step: Current simulation step.
    :param diagnostics: Timestep diagnostics used to evaluate burial conditions.
    :return: Tuple indicating whether a layer was created and the burial reason.
    """
    active_layer = layers[-1]
    reason = burial_reason(active_layer, params, diagnostics)
    if reason is None:
        return False, None

    active_layer["burial_step"] = step
    active_layer["burial_reason"] = reason
    layers.append(
        create_layer(
            thickness_mm=params.initial_mat_thickness_mm,
            biomass=params.new_layer_biomass_seed,
            eps=params.new_layer_eps_seed,
            sediment_mm=0.0,
            birth_step=step,
        )
    )
    return True, reason


In [ ]:
def layer_table(layers: list[dict], current_step: int) -> np.ndarray:
    """Convert the layer stack into a numeric table for plotting and export.

    :param layers: Current vertical stack of layer records from bottom to top.
    :param current_step: Current simulation step used to compute layer ages.
    :return: Two-dimensional array with layer properties.
    """
    rows = []
    cumulative_base = 0.0
    for index, layer in enumerate(layers):
        top = cumulative_base + layer["thickness_mm"]
        total_solids = layer["biomass"] + layer["eps"] + layer["sediment_mm"]
        sediment_fraction = layer["sediment_mm"] / total_solids if total_solids > 0 else 0.0
        rows.append([
            index,
            cumulative_base,
            top,
            layer["thickness_mm"],
            layer["biomass"],
            layer["eps"],
            layer["sediment_mm"],
            sediment_fraction,
            current_step - layer["birth_step"],
            -1 if layer["burial_step"] is None else layer["burial_step"],
        ])
        cumulative_base = top
    return np.array(rows, dtype=float)

In [ ]:
def summarise_layers(layers: list[dict], step: int) -> dict:
    """Summarise whole-column state at a timestep.

    :param layers: Current vertical stack of layer records.
    :param step: Current simulation step.
    :return: Summary metrics for the timestep.
    """
    total_height = sum(layer["thickness_mm"] for layer in layers)
    total_biomass = sum(layer["biomass"] for layer in layers)
    total_eps = sum(layer["eps"] for layer in layers)
    total_sediment = sum(layer["sediment_mm"] for layer in layers)
    return {
        "step": step,
        "height_mm": total_height,
        "layer_count": len(layers),
        "total_biomass": total_biomass,
        "total_eps": total_eps,
        "total_sediment_mm": total_sediment,
    }

In [ ]:
def run_1d_model(params: ModelParameters1D) -> tuple[list[dict], list[dict], list[np.ndarray]]:
    """Run the 1-D vertical-column stromatolite model.

    :param params: Model parameters controlling growth, sediment trapping, and burial.
    :return: Final layer stack, timestep history, and selected layer snapshots.
    """
    rng = np.random.default_rng(params.random_seed)
    layers = [
        create_layer(
            thickness_mm=params.initial_mat_thickness_mm,
            biomass=params.new_layer_biomass_seed,
            eps=params.new_layer_eps_seed,
            sediment_mm=0.0,
            birth_step=0,
        )
    ]
    history = []
    snapshots = []
    snapshot_steps = set(np.linspace(0, params.time_steps, 7, dtype=int))

    for step in range(params.time_steps + 1):
        summary = summarise_layers(layers, step)
        history.append(summary)
        if step in snapshot_steps:
            snapshots.append(layer_table(layers, step))
        if step == params.time_steps:
            break

        age_buried_layers(layers, params)
        column_height_mm = sum(layer["thickness_mm"] for layer in layers)
        diagnostics = update_active_layer(layers[-1], params, step, column_height_mm, rng)
        created_layer, reason = maybe_bury_active_layer(layers, params, step + 1, diagnostics)
        history[-1].update(diagnostics)
        history[-1]["created_layer"] = created_layer
        history[-1]["burial_reason"] = reason

    return layers, history, snapshots

In [ ]:
def history_as_array(history: list[dict], field: str) -> np.ndarray:
    """Extract a named history field as a NumPy array.

    :param history: List of timestep summary dictionaries.
    :param field: Dictionary key to extract from each summary.
    :return: Numeric array containing the requested field over time.
    """
    return np.array([row.get(field, np.nan) for row in history], dtype=float)

In [ ]:
def plot_growth_summary(history: list[dict], params: ModelParameters1D) -> None:
    """Plot height, layer count, and material totals through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "height_mm"), color="#1f5a8a")
    axes[0].set_title("Column height")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Height (mm)")

    axes[1].step(time_days, history_as_array(history, "layer_count"), where="post", color="#487a3b")
    axes[1].set_title("Burial events")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Layer count")

    axes[2].plot(time_days, history_as_array(history, "total_biomass"), label="biomass", color="#2b8c4a")
    axes[2].plot(time_days, history_as_array(history, "total_eps"), label="EPS", color="#79a941")
    axes[2].plot(time_days, history_as_array(history, "total_sediment_mm"), label="sediment", color="#b5925a")
    axes[2].set_title("Column materials")
    axes[2].set_xlabel("Time (days)")
    axes[2].legend(frameon=False)

    plt.savefig(IMAGE_DIR / "1d-growth-summary.png", dpi=300)

    plt.show()

In [ ]:
def plot_seasonal_forcing(history: list[dict], params: ModelParameters1D) -> None:
    """Plot seasonal forcing, water attenuation, and rate multipliers through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "incident_light"), label="incident", color="#d39c2f")
    axes[0].plot(time_days, history_as_array(history, "light_at_mat"), label="at mat", color="#1f5a8a")
    axes[0].set_title("Water-attenuated light")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Relative light")
    axes[0].legend(frameon=False)

    axes[1].plot(time_days, history_as_array(history, "water_depth_above_mat_mm"), color="#347f93")
    axes[1].set_title("Water above mat")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Depth (mm)")

    axes[2].plot(time_days, history_as_array(history, "temperature_c"), color="#b44e35")
    axes[2].set_title("Seasonal temperature")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("Temperature (deg C)")

    axes[3].plot(time_days, history_as_array(history, "light_factor"), label="light", color="#1f5a8a")
    axes[3].plot(time_days, history_as_array(history, "temperature_factor"), label="temperature", color="#487a3b")
    axes[3].set_title("Growth multipliers")
    axes[3].set_xlabel("Time (days)")
    axes[3].legend(frameon=False)

    plt.savefig(IMAGE_DIR / "1d-seasonal-forcing.png", dpi=300)

    plt.show()

In [ ]:
def plot_sediment_forcing(history: list[dict], params: ModelParameters1D) -> None:
    """Plot variable sediment supply components through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    storm_mask = history_as_array(history, "storm_today") > 0.0
    _, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "sediment_seasonal_multiplier"), color="#6b7f3a")
    axes[0].set_title("Seasonal sediment multiplier")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Multiplier")

    axes[1].plot(time_days, history_as_array(history, "sediment_supply_mm_per_day"), color="#9a7650", linewidth=1.0)
    if storm_mask.any():
        axes[1].scatter(
            time_days[storm_mask],
            history_as_array(history, "sediment_supply_mm_per_day")[storm_mask],
            color="#7f2f24",
            s=18,
            label="storm",
        )
        axes[1].legend(frameon=False)
    axes[1].set_title("Total sediment supply")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("mm/day")

    axes[2].plot(time_days, history_as_array(history, "supplied_sediment_mm"), label="arriving", color="#b5925a")
    axes[2].plot(time_days, history_as_array(history, "trapped_sediment_mm"), label="trapped", color="#487a3b")
    axes[2].set_title("Sediment trapping")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("mm/timestep")
    axes[2].legend(frameon=False)

    plt.savefig(IMAGE_DIR / "1d-sediment-forcing.png", dpi=300)

    plt.show()

In [ ]:
def plot_burial_events(history: list[dict], params: ModelParameters1D) -> None:
    """Plot sediment-driven burial events against sediment deposition.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to draw burial thresholds.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    created_mask = history_as_array(history, "created_layer") > 0.0
    _, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "trapped_sediment_mm"), color="#9a7650", linewidth=1.0)
    axes[0].axhline(params.rapid_burial_sediment_threshold_mm, color="#7f2f24", linestyle="--", linewidth=1.0)
    if created_mask.any():
        axes[0].scatter(
            time_days[created_mask],
            history_as_array(history, "trapped_sediment_mm")[created_mask],
            color="#7f2f24",
            s=20,
            label="burial",
        )
        axes[0].legend(frameon=False)
    axes[0].set_title("Rapid burial pulses")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Trapped sediment (mm/timestep)")

    axes[1].step(time_days, history_as_array(history, "layer_count"), where="post", color="#487a3b")
    axes[1].set_title("New laminae")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Layer count")

    plt.savefig(IMAGE_DIR / "1d-burial-events.png", dpi=300)

    plt.show()

In [ ]:
def plot_final_column(layer_data: np.ndarray) -> None:
    """Plot the final one-dimensional laminated column.

    :param layer_data: Numeric layer table created by layer_table.
    :return: None. A Matplotlib figure is displayed.
    """
    _, ax = plt.subplots(figsize=(3.5, 9), constrained_layout=True)
    for row in layer_data:
        _, base, top, thickness, biomass, eps, sediment, sediment_fraction, _, burial_step = row
        biological_signal = min(1.0, biomass + eps)
        color = (
            0.30 + 0.45 * sediment_fraction,
            0.52 + 0.28 * biological_signal,
            0.26 + 0.18 * (1.0 - sediment_fraction),
        )
        ax.bar(0, thickness, bottom=base, width=1.0, color=color, edgecolor="#2f2b22", linewidth=0.4)

    ax.set_xlim(-0.65, 0.65)
    ax.set_xticks([])
    ax.set_ylabel("Height above base (mm)")
    ax.set_title("Final laminated column")
    ax.set_facecolor("#f6f3ea")

    plt.savefig(IMAGE_DIR / "1d-final-column.png", dpi=300)
    plt.show()

In [ ]:
def plot_column_snapshots(snapshots: list[np.ndarray]) -> None:
    """Plot selected vertical-column snapshots through time.

    :param snapshots: List of layer tables captured during the model run.
    :return: None. A Matplotlib figure is displayed.
    """
    _, axes = plt.subplots(1, len(snapshots), figsize=(14, 5), sharey=True, constrained_layout=True)
    for ax, snapshot in zip(axes, snapshots):
        for row in snapshot:
            _, base, _, thickness, biomass, eps, sediment, sediment_fraction, _, _ = row
            biological_signal = min(1.0, biomass + eps)
            color = (
                0.30 + 0.45 * sediment_fraction,
                0.52 + 0.28 * biological_signal,
                0.26 + 0.18 * (1.0 - sediment_fraction),
            )
            ax.bar(0, thickness, bottom=base, width=0.9, color=color, edgecolor="#2f2b22", linewidth=0.3)
        step = int(snapshot[0, 8]) if snapshot.size else 0
        ax.set_title(f"snapshot {step}")
        ax.set_xlim(-0.7, 0.7)
        ax.set_xticks([])
    axes[0].set_ylabel("Height above base (mm)")

    plt.savefig(IMAGE_DIR / "1d-column-snapshots.png", dpi=300)
    plt.show()

In [ ]:
def save_layer_table(layer_data: np.ndarray, output_path: Path) -> None:
    """Save the final layer table as a CSV file.

    :param layer_data: Numeric layer table created by layer_table.
    :param output_path: Destination CSV path.
    :return: None. The CSV file is written to disk.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    header = "layer_index,base_mm,top_mm,thickness_mm,biomass,eps,sediment_mm,sediment_fraction,age_steps,burial_step"
    np.savetxt(output_path, layer_data, delimiter=",", header=header, comments="", fmt="%.6f")

## Run The Simulation

In [ ]:
params = load_1d_parameters(PARAMETER_FILE)
layers, history, snapshots = run_1d_model(params)
final_layers = layer_table(layers, params.time_steps)

print(f"Final height: {history[-1]['height_mm']:.2f} mm")
print(f"Layer count: {history[-1]['layer_count']}")
print(f"Mean lamina thickness: {final_layers[:, 3].mean():.2f} mm")

In [ ]:
plot_growth_summary(history, params)

In [ ]:
plot_seasonal_forcing(history, params)


In [ ]:
plot_sediment_forcing(history, params)


In [ ]:
plot_burial_events(history, params)


In [ ]:
plot_final_column(final_layers)

In [ ]:
plot_column_snapshots(snapshots)

## Save The Layered Output

In [ ]:
output_file = DATA_DIR / "1d-layers.csv"
save_layer_table(final_layers, output_file)
print(f"Saved {len(final_layers)} layers to {output_file}")